# Análisis de emociones en tiempo real

Importación de paquetes

In [1]:
import time

import cv2
import joblib
import mediapipe as mp
import numpy as np
from deepface import DeepFace
from tensorflow.keras.models import load_model


Variables

In [2]:
SVM_MODEL_PATH = "model_results/svm/eigenfaces_svm.joblib"
SVM_IMG_SIZE = 32

CNN_MODEL_PATH = "model_results/cnn/cnn_emotion.h5"
CNN_IMG_SIZE = 32
CNN_CLASSES = ["Angry", "Fear", "Happy", "Sad", "Suprise"]

CAM_INDEX = 0
BBOX_MARGIN = 0.2  # margen extra alrededor de la caja que forman los landmarks

svm_model = joblib.load(SVM_MODEL_PATH)
cnn_model = load_model(CNN_MODEL_PATH)


Landmarks y features geométricas (MediaPipe)

In [ ]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

latest_result = None


def _on_face_result(result, output_image, timestamp_ms):
    global latest_result
    latest_result = result


face_landmarker_options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=VisionRunningMode.LIVE_STREAM,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_tracking_confidence=0.5,
    result_callback=_on_face_result,
)
face_landmarker = FaceLandmarker.create_from_options(face_landmarker_options)

# Subconjunto de puntos que se dibujan sobre el vídeo
DRAW_LANDMARKS = [33, 133, 362, 263, 160, 158, 153, 144, 385, 387, 373, 380, 61, 291, 13, 14, 105, 55, 334, 285]


def _dist(p1, p2):
    return float(np.linalg.norm(np.asarray(p1) - np.asarray(p2)))


def compute_geometric_features(landmarks_px):
    def pt(i):
        return landmarks_px[i]

    iod = _dist(pt(33), pt(263))

    ear_a = (_dist(pt(160), pt(144)) + _dist(pt(158), pt(153))) / (2 * _dist(pt(33), pt(133)))
    ear_b = (_dist(pt(385), pt(380)) + _dist(pt(387), pt(373))) / (2 * _dist(pt(362), pt(263)))

    mar = _dist(pt(13), pt(14)) / _dist(pt(61), pt(291))
    mouth_width_norm = _dist(pt(61), pt(291)) / iod

    eyebrow_raise_a = _dist(pt(105), pt(158)) / iod
    eyebrow_raise_b = _dist(pt(334), pt(387)) / iod
    inter_eyebrow = _dist(pt(55), pt(285)) / iod

    mouth_mid_y = (pt(13)[1] + pt(14)[1]) / 2
    left_corner_lift = (mouth_mid_y - pt(61)[1]) / iod
    right_corner_lift = (mouth_mid_y - pt(291)[1]) / iod

    return np.array(
        [
            ear_a,
            ear_b,
            mar,
            mouth_width_norm,
            eyebrow_raise_a,
            eyebrow_raise_b,
            inter_eyebrow,
            left_corner_lift,
            right_corner_lift,
        ],
        dtype=np.float32,
    )


Detección de la cara para ambos modelos

In [4]:
def detect_face(frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    frame_h, frame_w = frame_bgr.shape[:2]

    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    face_landmarker.detect_async(mp_image, int(time.time() * 1000))

    if latest_result is None or not latest_result.face_landmarks:
        return None

    landmarks_px = np.array(
        [[lm.x * frame_w, lm.y * frame_h] for lm in latest_result.face_landmarks[0]],
        dtype=np.float32,
    )

    x_min, y_min = landmarks_px.min(axis=0)
    x_max, y_max = landmarks_px.max(axis=0)
    mx = (x_max - x_min) * BBOX_MARGIN
    my = (y_max - y_min) * BBOX_MARGIN

    x = max(int(x_min - mx), 0)
    y = max(int(y_min - my), 0)
    w = min(int(x_max + mx), frame_w) - x
    h = min(int(y_max + my), frame_h) - y

    return landmarks_px, (x, y, w, h)


Predicción con DeepFace

In [5]:
def predict_deepface(face_bgr):
    results = DeepFace.analyze(
        img_path=face_bgr,
        actions=["emotion"],
        detector_backend="skip",
        enforce_detection=False,
    )
    return f"DeepFace: {results[0]['dominant_emotion']}"


Predicción con SVM + Eigenfaces + keypoints

In [6]:
def predict_svm(face_bgr, geo_features):
    face_gray = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
    face_resized = cv2.resize(face_gray, (SVM_IMG_SIZE, SVM_IMG_SIZE))
    pixels = face_resized.flatten().astype(np.float32)
    vector = np.concatenate([pixels, geo_features]).reshape(1, -1)
    return f"SVM: {svm_model.predict(vector)[0]}"


Predicción con CNN + keypoints

In [7]:
def predict_cnn(face_bgr, geo_features):
    face_gray = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2GRAY)
    face_resized = cv2.resize(face_gray, (CNN_IMG_SIZE, CNN_IMG_SIZE))
    img_input = face_resized.astype(np.float32).reshape(1, CNN_IMG_SIZE, CNN_IMG_SIZE, 1) / 255.0
    geo_input = geo_features.reshape(1, -1)

    pred = cnn_model.predict([img_input, geo_input], verbose=0)
    return f"CNN: {CNN_CLASSES[np.argmax(pred[0])]}"


Botón para alternar de modelo

In [8]:
BUTTON_TOPLEFT = (10, 10)
BUTTON_BOTTOMRIGHT = (190, 50)

MODEL_NAMES = ["DeepFace", "SVM (Eigenfaces)", "CNN"]
current_model = {"name": MODEL_NAMES[0]}


def is_inside_button(x, y):
    (bx1, by1), (bx2, by2) = BUTTON_TOPLEFT, BUTTON_BOTTOMRIGHT
    return bx1 <= x <= bx2 and by1 <= y <= by2


def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and is_inside_button(x, y):
        next_idx = (MODEL_NAMES.index(current_model["name"]) + 1) % len(MODEL_NAMES)
        current_model["name"] = MODEL_NAMES[next_idx]


def draw_button(frame_bgr):
    (bx1, by1), (bx2, by2) = BUTTON_TOPLEFT, BUTTON_BOTTOMRIGHT
    cv2.rectangle(frame_bgr, (bx1, by1), (bx2, by2), (50, 50, 50), -1)
    cv2.rectangle(frame_bgr, (bx1, by1), (bx2, by2), (255, 255, 255), 1)
    cv2.putText(
        frame_bgr,
        current_model["name"],
        (bx1 + 8, by1 + 27),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.5,
        (255, 255, 255),
        1,
        cv2.LINE_AA,
    )


Prueba final de webcam

In [ ]:
WINDOW_NAME = "Deteccion de emociones (DeepFace vs SVM vs CNN)"

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la webcam")

cv2.namedWindow(WINDOW_NAME)
cv2.setMouseCallback(WINDOW_NAME, on_mouse)

try:
    while True:
        ok, frame_bgr = cap.read()
        if not ok:
            break

        detection = detect_face(frame_bgr)
        if detection is not None:
            landmarks_px, (x, y, w, h) = detection
            face_bgr = frame_bgr[y : y + h, x : x + w]

            try:
                geo_features = compute_geometric_features(landmarks_px)

                if current_model["name"] == "DeepFace":
                    label = predict_deepface(face_bgr)
                elif current_model["name"] == "SVM (Eigenfaces)":
                    label = predict_svm(face_bgr, geo_features)
                else:
                    label = predict_cnn(face_bgr, geo_features)

                cv2.rectangle(frame_bgr, (x, y), (x + w, y + h), (0, 255, 0), 2)
                for idx in DRAW_LANDMARKS:
                    kx, ky = landmarks_px[idx]
                    cv2.circle(frame_bgr, (int(kx), int(ky)), 2, (0, 0, 255), -1)

                cv2.putText(
                    frame_bgr,
                    label,
                    (x, max(y - 10, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA,
                )
            except Exception:
                # El recorte quedó demasiado pequeño, vacío o el modelo falló en este frame
                pass

        draw_button(frame_bgr)
        cv2.imshow(WINDOW_NAME, frame_bgr)

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q") or key == 27:
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    face_landmarker.close()
